In [1]:
from __future__ import (absolute_import, division,
                        print_function, unicode_literals)

import warnings
warnings.simplefilter('ignore')

# general purpose packages
import pandas as pd
import numpy as np
import os
import json
import time
import re
import csv
import subprocess
import sys

import scipy.stats as stats
import statsmodels.stats as smstats
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import sklearn

import umap

from multiprocessing import Process, Manager, Pool
import multiprocessing
from functools import partial

from collections import Counter

import seaborn as sns; sns.set()

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
matplotlib.rcParams['backend'] = "Qt5Agg"
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter

from IPython.display import display, Image

from adjustText import adjust_text
import builtins
%matplotlib inline

# for working with sam/bam files
import HTSeq

# for working with sam/bam files
import HTSeq

# for working with yaml files
import ruamel.yaml

import itertools

In [2]:
# paths to subdirectories
subdirs = {}

subdirs['main_project_dir'] = '/scicore/home/zavolan/GROUP/Primer_Probe_design/'
subdirs['human_annotation_dir'] = '/scicore/home/zavolan/GROUP/Genomes/homo_sapiens/'
subdirs['swine_annotation_dir'] = '/scicore/home/zavolan/GROUP/Genomes/swine/'

subdirs['shared_project_dir'] = subdirs['main_project_dir']
subdirs['PRRSV_dir'] = subdirs['shared_project_dir']+'PRRSV/'
subdirs['PRRSV_github_dir'] = subdirs['PRRSV_dir']+'github/'
subdirs['PRRSV_materials_dir'] = subdirs['PRRSV_dir']+'materials/'
subdirs['PRRSV_reference_genomes_dir'] = subdirs['PRRSV_dir']+'reference_genomes/'

# technicals
subdirs['temp_dir'] = subdirs['shared_project_dir']+'temp/'
subdirs['figures_dir'] = subdirs['shared_project_dir']+'figures/'
subdirs['tables_dir'] = subdirs['shared_project_dir']+'tables/'
subdirs['scripts_dir'] = subdirs['shared_project_dir']+'scripts/'
subdirs['apptainer_dir'] = subdirs['shared_project_dir']+'apptainer/'
subdirs['slurm_dir'] = subdirs['shared_project_dir']+'slurm/'

# paths to files
file_paths = {}
### genome annotation files
file_paths['human_genome_file'] = subdirs['human_annotation_dir']+'GRCh38.primary_assembly.genome.fa'
file_paths['human_genome_fai_file'] = subdirs['human_annotation_dir']+'GRCh38.primary_assembly.genome.fa.fai'
file_paths['human_annotation_file'] = subdirs['human_annotation_dir']+'hg38_v42/gencode.v42.annotation.gtf'
file_paths['human_RNAcentral_annotation_file'] = subdirs['human_annotation_dir']+'hg38_v42/homo_sapiens.GRCh38.gff3.gz'
file_paths['human_enriched_annotation_file'] = subdirs['human_annotation_dir']+'hg38_v42/enriched.gencode.v42.annotation.gtf'
file_paths['human_chrom_sizes'] = subdirs['human_annotation_dir']+'hg38.chrom.sizes'

os.system('mkdir -p '+' '.join(list(subdirs.values()))) # create all subdirs

0

# PRRSV - track hub

In [ ]:
# create conda env from yml file
# from current directory:
# conda env create -f ./envs/primer_probe_design.yaml

## Initial github configuration

In [3]:
subdirs['PRRSV_github_dir']

'/scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/'

In [37]:
# cd to subdirs['PRRSV_github_dir']
# git init
# then create github repo on the github website
# then create test file

# then, for every change to put to the repo online:
# git add -A; git commit -m "message"; git push -u origin main

## reference genomes for PRRSV-type 1 (Lelystad) and PRRSV-type 2 (VR-2332)

In [ ]:
# this is hub URL on github
# https://raw.githubusercontent.com/zavolanlab/primer_probe_design/refs/heads/main/hub.txt
# this is direct link for UCSC
# https://genome.ucsc.edu/cgi-bin/hgTracks?genome=prrsv_lelystad_vr2332&hubUrl=https://raw.githubusercontent.com/zavolanlab/primer_probe_design/refs/heads/main/hub.txt

In [77]:
# the following command uses esearch command from edirect package of NIH: https://www.ncbi.nlm.nih.gov/books/NBK179288/

# download the files of interest from GENBANK using GENBANKD ids, e.g M96262
command = 'esearch -db nucleotide -query "'+'M96262'+'" | efetch -format fasta > '+subdirs['PRRSV_reference_genomes_dir']+'M96262.Lelystad.PRRSV_type1.fasta'+'; '
command = command+'esearch -db nucleotide -query "'+'M96262'+'" | efetch -format gb > '+subdirs['PRRSV_reference_genomes_dir']+'M96262.Lelystad.PRRSV_type1.gb'+'; '
command = command+'esearch -db nucleotide -query "'+'U87392'+'" | efetch -format fasta > '+subdirs['PRRSV_reference_genomes_dir']+'U87392.VR2332.PRRSV_type2.fasta'+'; '
command = command+'esearch -db nucleotide -query "'+'U87392'+'" | efetch -format gb > '+subdirs['PRRSV_reference_genomes_dir']+'U87392.VR2332.PRRSV_type2.gb'+'; '
command = command+'cat '+subdirs['PRRSV_reference_genomes_dir']+'M96262.Lelystad.PRRSV_type1.fasta '+subdirs['PRRSV_reference_genomes_dir']+'U87392.VR2332.PRRSV_type2.fasta > '+subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.fasta'
command

'esearch -db nucleotide -query "M96262" | efetch -format fasta > /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/M96262.Lelystad.PRRSV_type1.fasta; esearch -db nucleotide -query "M96262" | efetch -format gb > /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/M96262.Lelystad.PRRSV_type1.gb; esearch -db nucleotide -query "U87392" | efetch -format fasta > /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/U87392.VR2332.PRRSV_type2.fasta; esearch -db nucleotide -query "U87392" | efetch -format gb > /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/U87392.VR2332.PRRSV_type2.gb; cat /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/M96262.Lelystad.PRRSV_type1.fasta /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/U87392.VR2332.PRRSV_type2.fasta > /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/PRRSV_reference_genomes.fasta'

In [78]:
# change chromosome names in the file "PRRSV_reference_genomes.fasta" to "Lelystad.PRRSV_1" and "VR2332.PRRSV_2"
# conda activate primer_probe_design
input_fasta = subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.fasta'
renaming_file = subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.renaming.txt'
output_fasta = subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.renamed.fasta'

paths_tsv = pd.DataFrame([['M96262.2','Lelystad.PRRSV_1'],['U87392.3','VR2332.PRRSV_2']],columns=['cur_id','new_id'])
paths_tsv.to_csv(renaming_file, sep=str('\t'),header=False,index=None)

command = """seqkit replace -p '^(\S+)(.+?)$' -r '{kv}$2' -k """+renaming_file+' '+input_fasta+' > '+output_fasta
print(command)

seqkit replace -p '^(\S+)(.+?)$' -r '{kv}$2' -k /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/PRRSV_reference_genomes.renaming.txt /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/PRRSV_reference_genomes.fasta > /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/PRRSV_reference_genomes.renamed.fasta


In [79]:
# parse genbank files to
from Bio import SeqIO

def get_feature_records_df_from_GB(file_path):
    records = SeqIO.parse(file_path, format="gb")
    a = []
    for record in records:
        for feature in record.features:
            if feature.location:
                start = int(feature.location.start)
                end = int(feature.location.end)
                strand = ('+' if feature.location.strand==1 else '-')
                feature_type = feature.type
                note,product = None,None
                if feature.qualifiers:
                    if 'note' in feature.qualifiers.keys():
                        note = feature.qualifiers['note'][0]
                    if 'product' in feature.qualifiers.keys():
                        product = feature.qualifiers['product'][0]
                a.append([start,end,strand,feature_type,note,product])
    df = pd.DataFrame(a,columns=['start','end','strand','feature_type','note','product'])
    return df

In [106]:
gb_files = ['M96262.Lelystad.PRRSV_type1.gb','U87392.VR2332.PRRSV_type2.gb']
chromosome_names = ['Lelystad.PRRSV_1','VR2332.PRRSV_2']
a = []
for i,gb_file in enumerate(gb_files):
    full_file_path = subdirs['PRRSV_reference_genomes_dir']+gb_file
    df = get_feature_records_df_from_GB(full_file_path)
    df['chr'] = chromosome_names[i]
    a.append(df)
annotation_df = pd.concat(a).reset_index(drop=True)

# make gtf file in the GENCODE format
gtf = annotation_df.copy()
gtf = gtf.loc[gtf['feature_type']!='source'].reset_index(drop=True)

def get_gene_id(x):
    if x['feature_type']=='CDS':
        if (x['note'] is not None) and x['note'].startswith('ORF'):
            return x['note'].split('; ')[0]
        else:
            if x['product'].startswith('ORF'):
                return x['product']
            else:
                return ''
    else:
        if (x['note'] is not None) and ('region of minus 1 ribosomal frameshift' in x['note']):
            return 'Ribo_minus_one_frameshift'
        else:
            return x['feature_type'].replace("'",'_')

def get_gene_name(x):
    if x['feature_type']=='CDS':
        if (x['gene_id'] == 'ORF1a'):
            return '1a replicase'
        elif (x['gene_id'] == 'ORF1b'):
            return '1b replicase'
        elif (x['gene_id'] == 'ORF1ab'):
            return '1ab replicase'
        else:
            return x['product']
    else:
        return x['gene_id']
            
gtf['gene_id'] = gtf.apply(lambda x:get_gene_id(x),1)
gtf['gene_name'] = gtf.apply(lambda x:get_gene_name(x),1)

gtf[0] = gtf['chr']
gtf[1] = 'genbank'
gtf[2] = 'gene'
gtf[3] = gtf['start']
gtf[4] = gtf['end']
gtf[5] = '.'
gtf[6] = gtf['strand']
gtf[7] = 0
gtf[8] = 'gene_id "'+gtf['gene_id']+'"; gene_name "'+gtf['gene_name']+'"; gene_biotype "'+gtf['feature_type']+'"'

gtf[list(range(0,9))].to_csv(subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.gtf',sep=str('\t'),header=False,index=None,quoting=csv.QUOTE_NONE)

In [120]:
# now, prepare annotation in genePred format, compatible with UCSC
genePred_df = gtf.copy()
genePred_df[0] = genePred_df['gene_id']
genePred_df[1] = genePred_df['chr']
genePred_df[2] = genePred_df['strand']
genePred_df[3] = genePred_df['start']
genePred_df[4] = genePred_df['end']
genePred_df[5] = genePred_df.apply(lambda x:(x['start'] if x['feature_type']=='CDS' else 0),1)
genePred_df[6] = genePred_df.apply(lambda x:(x['end'] if x['feature_type']=='CDS' else 0),1)
genePred_df[7] = 1
genePred_df[8] = genePred_df['start']
genePred_df[9] = genePred_df['end']
genePred_df[10] = 0
genePred_df[11] = genePred_df['gene_name']
genePred_df[12] = genePred_df.apply(lambda x:('cmpl' if x['feature_type']=='CDS' else 'unk'),1)
genePred_df[13] = genePred_df.apply(lambda x:('cmpl' if x['feature_type']=='CDS' else 'unk'),1)
genePred_df[14] = genePred_df.apply(lambda x:(0 if x['feature_type']=='CDS' else -1),1)
genePred_df[range(0,15)].to_csv(subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.genePred',sep=str('\t'),header=False,index=None,quoting=csv.QUOTE_NONE)

In [121]:
# prepare genome and put to github for TrackHub visualization
ref_genome_title = 'prrsv_lelystad_vr2332'
github_genome_dir = subdirs['PRRSV_github_dir']+ref_genome_title+'/'
command = 'mkdir -p '+github_genome_dir
out = subprocess.check_output(command, shell=True)

command = 'faToTwoBit '+subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.renamed.fasta'+' '+github_genome_dir+ref_genome_title+'.2bit'
out = subprocess.check_output(command, shell=True)

command = 'twoBitInfo '+github_genome_dir+ref_genome_title+'.2bit'+' stdout | sort -k2rn > '+github_genome_dir+ref_genome_title+'.chrom.sizes'
out = subprocess.check_output(command, shell=True)

# prepare chromosome diagram
command = 'cat '+github_genome_dir+ref_genome_title+'.chrom.sizes'+""" | sort -k1,1 -k2,2n | awk '{print $1,0,$2,$1,"gneg"}' > """+github_genome_dir+'cytoBandIdeo.bed'
out = subprocess.check_output(command, shell=True)
out = subprocess.check_output(command, shell=True)
command = 'bedToBigBed -type=bed4 '+github_genome_dir+'cytoBandIdeo.bed'+' -as='+github_genome_dir+'cytoBand.as'+' '+github_genome_dir+ref_genome_title+'.chrom.sizes'+' '+github_genome_dir+'cytoBandIdeo.bigBed'
out = subprocess.check_output(command, shell=True)

# prepare gene annotation

command = 'genePredToBigGenePred '+subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.genePred'+' '+github_genome_dir+'PRRSV_reference_genomes.BigGenePred'
out = subprocess.check_output(command, shell=True)

command = 'bedToBigBed -type=bed12+8 -tab -as='+github_genome_dir+'bigGenePred.as '+github_genome_dir+'PRRSV_reference_genomes.BigGenePred '+github_genome_dir+ref_genome_title+'.chrom.sizes'+' '+github_genome_dir+ref_genome_title+'.gene_annotation.bb'
out = subprocess.check_output(command, shell=True)

pass1 - making usageList (2 chroms): 1 millis
pass2 - checking and writing primary data (2 records, 5 fields): 0 millis
pass1 - making usageList (2 chroms): 0 millis
pass2 - checking and writing primary data (21 records, 20 fields): 1 millis


## US Swine pathogen database

In [112]:
# strain query

records_df = pd.read_csv(subdirs['PRRSV_materials_dir']+'PRRSV_type2.29_08_2025.US-SPD.strain_query.csv',delimiter=",",index_col=None,header=0)


In [120]:
list(records_df.loc[~records_df['Strain or Isolate Name'].isna()]['Strain or Isolate Name'].unique())[:30]

['vs HP PRRSV MN1',
 'PRRSV Peru/2017/10',
 'PRRSV Peru/2017/8',
 'PRRSV Peru/2017/7',
 'PRRSV Peru/2017/6',
 'PRRSV Peru/2016/20',
 'PRRSV Peru/2016/16',
 'PRRSV Peru/2016/9',
 'PRRSV Peru/2016/5',
 'PRRSV Peru/2016/4',
 'PRRSV Peru/2016/3',
 'PRRSV Peru/2016/2',
 'PRRSV Peru/2016/1',
 'PRRSV Peru/2015/19',
 'PRRSV Peru/2015/18',
 'PRRSV Peru/2015/17',
 'PRRSV Peru/2015/15',
 'PRRSV Peru/2015/14',
 'PRRSV Peru/2015/13',
 'PRRSV Peru/2015/12',
 'PRRSV Peru/2015/11',
 'EDRD-1',
 'P129',
 'PL97-1/LP1',
 'VR-2332',
 'Jam2',
 'Jos1',
 'Jis1',
 'Jis2',
 'Jsi1']

In [11]:
# I extracted all records from the database: https://swinepathogendb.org/sample_search/PRRSV-2
# I selected "protein" in advanced output options 
records_df = pd.read_csv(subdirs['PRRSV_materials_dir']+'PRRSV_type2.18_08_2025.US-SPD.records.txt',delimiter="|",index_col=None,header=None)
records_df = records_df[[0,2,11,13]]
records_df.columns = ['USSPD_ID','GB_ID','annotation','subseq_coords']
records_df = records_df.loc[records_df['annotation'].str.contains(' full')].reset_index(drop=True)

In [160]:
# try downloading respective nt seqs from GB to single fasta file

target_path = subdirs['PRRSV_materials_dir']+'USSPD_collection/PRRSV_type2.dna.fasta'

command = 'mkdir -p '+subdirs['PRRSV_materials_dir']+'USSPD_collection/'
out = subprocess.check_output(command, shell=True)
records_df[['GB_ID']].drop_duplicates().to_csv(subdirs['PRRSV_materials_dir']+'USSPD_collection/accessions.txt',sep=str('\t'),header=False,index=None,quoting=csv.QUOTE_NONE)

command = 'cat '+subdirs['PRRSV_materials_dir']+'USSPD_collection/accessions.txt'+' | efetch -db nucleotide -format fasta > '+target_path
print(command)

cat /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/USSPD_collection/accessions.txt | efetch -db nucleotide -format fasta > /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/USSPD_collection/PRRSV_type2.dna.fasta


In [5]:
###
# map with minimap2
###

# conda activate primer_probe_design

ref_genome_title = 'prrsv_lelystad_vr2332'
github_genome_dir = subdirs['PRRSV_github_dir']+ref_genome_title+'/'

fasta_input_dir = subdirs['PRRSV_materials_dir']+'USSPD_collection/'
bam_output_dir = github_genome_dir
command = 'mkdir -p '+bam_output_dir
out = subprocess.check_output(command, shell=True)

genome_file = subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.renamed.fasta'
fasta_input_file = fasta_input_dir+'PRRSV_type2.dna.fasta'
bam_output_file = bam_output_dir+'USSPD_collection.sorted.bam'
bed_output_file = bam_output_dir+'USSPD_collection.sorted.bed'

threads = 8

command = ''
command = command+'minimap2 -t '+str(threads)+' -ax map-ont -k7 -L --MD -2 -Q -secondary=no --sam-hit-only '+genome_file+' '+fasta_input_file+' | '+\
'samtools view -@ '+str(threads)+' -q 1 -F 0x100 -u | samtools sort -@ '+str(threads)+' - -o '+bam_output_file+' && '+\
'samtools index -@ '+str(threads)+' '+bam_output_file+'; '
command

'minimap2 -t 8 -ax map-ont -k7 -L --MD -2 -Q -secondary=no --sam-hit-only /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/PRRSV_reference_genomes.renamed.fasta /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/USSPD_collection/PRRSV_type2.dna.fasta | samtools view -@ 8 -q 1 -F 0x100 -u | samtools sort -@ 8 - -o /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/USSPD_collection.sorted.bam && samtools index -@ 8 /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/USSPD_collection.sorted.bam; '

In [52]:
command = ''
command = command+'bedtools bamtobed -cigar -i '+bam_output_file+' > '+bed_output_file
command

'bedtools bamtobed -cigar -i /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/USSPD_collection.sorted.bam > /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/USSPD_collection.sorted.bed'

In [90]:
bed_df = pd.read_csv(bed_output_file,delimiter="\t",index_col=None,header=None)
bed_df.columns=['chr','start','end','GB_ID','MAPQ','strand','cigar']

bed_df = bed_df.loc[bed_df['chr']=='VR2332.PRRSV_2'].reset_index(drop=True) # we've used only records for PRRSV type 2 !!!

bed_df = pd.merge(bed_df,records_df,how='inner',on='GB_ID')

bed_df['relative_annotation_start'] = bed_df['subseq_coords'].str.split(', ',expand=True)[0].astype('int')
bed_df['relative_annotation_end'] = bed_df['subseq_coords'].str.split(', ',expand=True)[1].astype('int')

cigars = bed_df[['strand','cigar','relative_annotation_start','relative_annotation_end']].drop_duplicates().reset_index(drop=True)
print(len(cigars), len(bed_df))

cigar_pattern = re.compile(r'(\d+)([MIDNSHPX=])')
# we need to look only at cigar elements that "consume" reference
reference_consuming_elements = ['M','D','N','=','X']
offset_consuming_elements = ['M','=','S','X','I']

def get_shifts(x,reference_consuming_elements,offset_consuming_elements):
    shifts = []
    offsets = [x['relative_annotation_start'],x['relative_annotation_end']]
    cigar_elements = cigar_pattern.findall(x['cigar'])
    
    for offset in offsets:
        shift = 0
        consumed_nts = 0
        reference_shift = 0
    
        if x['strand']=='-':
            cigar_elements = cigar_elements[::-1]
        for length_str, op_type in cigar_elements:
            length_str = int(length_str)
            if op_type in offset_consuming_elements:
                if length_str+consumed_nts<offset:
                    consumed_nts = consumed_nts+length_str
                else:
                    # need to stop at this element
                    if op_type in reference_consuming_elements:
                        shift = reference_shift+(offset-consumed_nts)
                    else:
                        shift = -1 # not valid, relative position ends in the part which is not mapped to the reference
                    break
            if op_type in reference_consuming_elements:
                reference_shift = reference_shift+length_str
        shifts.append(str(shift))
    return '|'.join(shifts)
    
cigars['shifts'] = cigars.apply(lambda x:get_shifts(x,reference_consuming_elements,offset_consuming_elements),1)
cigars['shift_start'] = cigars['shifts'].str.split('|',expand=True)[0].astype('int')
cigars['shift_end'] = cigars['shifts'].str.split('|',expand=True)[1].astype('int')
cigars = cigars.loc[(cigars['shift_start']!=-1)&(cigars['shift_end']!=-1)].reset_index(drop=True)

bed_df = pd.merge(bed_df,cigars,how='inner',on=['strand','cigar','relative_annotation_start','relative_annotation_end'])

bed_df['genomic_annotation_start'] = bed_df['start']+bed_df['shift_start']
bed_df['genomic_annotation_end'] = bed_df['start']+bed_df['shift_end']
bed_df['incompatible'] = (bed_df['start']>bed_df['genomic_annotation_start'])|(bed_df['end']<bed_df['genomic_annotation_end'])

print(Counter(bed_df['incompatible']))

bed_df = bed_df.loc[~bed_df['incompatible']].reset_index(drop=True) # do not retain incompatible alignments

bed_df['t']=1
gr = bed_df.groupby(['annotation','chr','genomic_annotation_start','genomic_annotation_end','strand']).agg({'t':np.sum}).reset_index().sort_values(['annotation','t'],ascending=[True,False]).reset_index(drop=True)
gr = pd.merge(gr,gr.groupby('annotation').agg({'t':np.sum}).reset_index().rename(columns={'t':'t_sum'}),how='inner',on='annotation')
gr['%']=gr['t']/gr['t_sum']*100

gr = gr.loc[gr['%']>1].reset_index(drop=True) # only retain isoforms occuring at >1%
gr['rank'] = gr.groupby('annotation')['%'].rank(method='average',ascending=True,)

43786 80774
Counter({False: 74827})


In [98]:
# now, make annotation in genePred format, compatible with UCSC
genePred_df = gr.copy()
genePred_df[0] = genePred_df['annotation']
genePred_df[1] = genePred_df['chr']
genePred_df[2] = genePred_df['strand']
genePred_df[3] = genePred_df['genomic_annotation_start']-1 # need to adjust for some reason; bed-like format
genePred_df[4] = genePred_df['genomic_annotation_end']
genePred_df = genePred_df.loc[genePred_df[4]>=genePred_df[3]].reset_index(drop=True)

genePred_df[5] = genePred_df[3] # we had CDS sequences annotation from USS
genePred_df[6] = genePred_df[4] # we had CDS sequences annotation from USS

genePred_df[7] = 1 # Number of exons
genePred_df[8] = genePred_df[3] # Exon start positions
genePred_df[9] = genePred_df[4] # Exon end positions

genePred_df[10] = genePred_df['rank'].astype('int') # score
genePred_df[11] = genePred_df['annotation'] # Alternate name

genePred_df[12] = 'cmpl' # Status of CDS start annotation
genePred_df[13] = 'cmpl' # Status of CDS end annotation
genePred_df[14] = 0 # Exon frame offsets
genePred_df.sort_values([1,3,4])[range(0,15)].to_csv(subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.USSPD_annotation.genePred',sep=str('\t'),header=False,index=None,quoting=csv.QUOTE_NONE)

In [99]:
# add to github a UCSC-compatible track

ref_genome_title = 'prrsv_lelystad_vr2332'
github_genome_dir = subdirs['PRRSV_github_dir']+ref_genome_title+'/'
command = 'mkdir -p '+github_genome_dir
out = subprocess.check_output(command, shell=True)

command = 'genePredToBigGenePred '+subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.USSPD_annotation.genePred'+' '+github_genome_dir+'PRRSV_reference_genomes.USSPD_annotation.BigGenePred'
out = subprocess.check_output(command, shell=True)

command = 'bedToBigBed -type=bed12+8 -tab -as='+github_genome_dir+'bigGenePred.as '+github_genome_dir+'PRRSV_reference_genomes.USSPD_annotation.BigGenePred '+github_genome_dir+ref_genome_title+'.chrom.sizes'+' '+github_genome_dir+ref_genome_title+'.USSPD_annotation.bb'
out = subprocess.check_output(command, shell=True)

pass1 - making usageList (1 chroms): 0 millis
pass2 - checking and writing primary data (26 records, 20 fields): 20 millis


## WGS data

### download

In [3]:
# canadian WGS of PPRSV-type2
paths_tsv = pd.read_csv(subdirs['PRRSV_materials_dir']+'WGS_PMID32817228/WGS_Canada.tsv',delimiter="\t",index_col=None,header=0)
paths_tsv['sample'] = paths_tsv['GenBank']+'.'+paths_tsv['sample_type']+'.'+paths_tsv['number_of_pooled_animals'].str.replace('.','')
paths_tsv['target_path'] = subdirs['PRRSV_materials_dir']+'WGS_PMID32817228/'+paths_tsv['sample']+'.fasta'

In [4]:
# try downloading everything to single fasta file

target_path = subdirs['PRRSV_materials_dir']+'WGS_PMID32817228/whole_genome_seqs.PRRSV_2.fasta'

command = ''
for index,row in paths_tsv.iterrows():
    command = command+'esearch -db nucleotide -query "'+row['GenBank']+'" | efetch -format fasta '+('>' if index==0 else '>>')+' '+target_path+'; '    
command

'esearch -db nucleotide -query "MN865482" | efetch -format fasta > /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/WGS_PMID32817228/whole_genome_seqs.PRRSV_2.fasta; esearch -db nucleotide -query "MN865566" | efetch -format fasta >> /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/WGS_PMID32817228/whole_genome_seqs.PRRSV_2.fasta; esearch -db nucleotide -query "MN865567" | efetch -format fasta >> /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/WGS_PMID32817228/whole_genome_seqs.PRRSV_2.fasta; esearch -db nucleotide -query "MN865483" | efetch -format fasta >> /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/WGS_PMID32817228/whole_genome_seqs.PRRSV_2.fasta; esearch -db nucleotide -query "MN865484" | efetch -format fasta >> /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/WGS_PMID32817228/whole_genome_seqs.PRRSV_2.fasta; esearch -db nucleotide -query "MN865485" | efetch -format fasta >> /scicore/home/zavolan/GROUP/

In [5]:
# rename the ids in the fasta file
# conda activate primer_probe_design

input_fasta = subdirs['PRRSV_materials_dir']+'WGS_PMID32817228/whole_genome_seqs.PRRSV_2.fasta'
renaming_file = subdirs['PRRSV_materials_dir']+'WGS_PMID32817228/whole_genome_seqs.PRRSV_2.renaming.txt'
output_fasta = subdirs['PRRSV_materials_dir']+'WGS_PMID32817228/whole_genome_seqs.PRRSV_2.renamed.fasta'

paths_tsv['GenBank_id'] = paths_tsv['GenBank'].astype('str')+'.1'
paths_tsv[['GenBank_id','sample']].to_csv(renaming_file, sep=str('\t'),header=False,index=None)

command = """seqkit replace -p '^(\S+)(.+?)$' -r '{kv}$2' -k """+renaming_file+' '+input_fasta+' > '+output_fasta
print(command)

seqkit replace -p '^(\S+)(.+?)$' -r '{kv}$2' -k /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/WGS_PMID32817228/whole_genome_seqs.PRRSV_2.renaming.txt /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/WGS_PMID32817228/whole_genome_seqs.PRRSV_2.fasta > /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/WGS_PMID32817228/whole_genome_seqs.PRRSV_2.renamed.fasta


### pairwise alignment

In [92]:
###
# map with minimap2
###

# conda activate primer_probe_design

ref_genome_title = 'prrsv_lelystad_vr2332'
github_genome_dir = subdirs['PRRSV_github_dir']+ref_genome_title+'/'

fasta_input_dir = subdirs['PRRSV_materials_dir']+'WGS_PMID32817228/'
bam_output_dir = github_genome_dir
command = 'mkdir -p '+bam_output_dir
out = subprocess.check_output(command, shell=True)

genome_file = subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.renamed.fasta'
chrom_sizes_file = github_genome_dir+ref_genome_title+'.chrom.sizes'

fasta_input_file = fasta_input_dir+'whole_genome_seqs.PRRSV_2.renamed.fasta'
bam_output_file = bam_output_dir+'WGS_Canadian_PMID32817228.sorted.bam'
sam_output_file = bam_output_dir+'WGS_Canadian_PMID32817228.sorted.sam'
paf_output_file = bam_output_dir+'WGS_Canadian_PMID32817228.sorted.paf'
maf_output_file = bam_output_dir+'WGS_Canadian_PMID32817228.sorted.maf'
bigmaf_output_file = bam_output_dir+'WGS_Canadian_PMID32817228.sorted.bigMaf.txt'
maf_bb_output_file = bam_output_dir+'WGS_Canadian_PMID32817228.sorted.maf.bb'

bed_output_file = bam_output_dir+'WGS_Canadian_PMID32817228.sorted.bed'
rnaseqmut_output_file = bam_output_dir+'WGS_Canadian_PMID32817228.rnaseqmut.txt'
bedgraph_output_file = bam_output_dir+'WGS_Canadian_PMID32817228.bg'
bigwig_output_file = bam_output_dir+'WGS_Canadian_PMID32817228.bigwig'

threads = 8

command = ''
command = command+'minimap2 -t '+str(threads)+' -ax map-ont -k7 -L --MD -2 -Q -secondary=no --sam-hit-only '+genome_file+' '+fasta_input_file+' | '+\
'samtools view -@ '+str(threads)+' -q 1 -F 0x100 -u | samtools sort -@ '+str(threads)+' - -o '+bam_output_file+' && '+\
'samtools index -@ '+str(threads)+' '+bam_output_file+'; '
command

'minimap2 -t 8 -ax map-ont -k7 -L --MD -2 -Q -secondary=no --sam-hit-only /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/PRRSV_reference_genomes.renamed.fasta /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/WGS_PMID32817228/whole_genome_seqs.PRRSV_2.renamed.fasta | samtools view -@ 8 -q 1 -F 0x100 -u | samtools sort -@ 8 - -o /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/WGS_Canadian_PMID32817228.sorted.bam && samtools index -@ 8 /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/WGS_Canadian_PMID32817228.sorted.bam; '

In [116]:
###
# use rnaseqmut to extract mutations relative to reference
###

command = """apptainer exec --bind """+subdirs['PRRSV_dir']+' '+subdirs['apptainer_dir']+'rnaseqmut_latest.sif '+\
'rnaseqmut --ref_fasta '+genome_file+' --min_read 1 --with_indel_read --with_indel --max_mismatch 100000000 --mut_span 1 '+bam_output_file+' > '+rnaseqmut_output_file
out = subprocess.check_output(command, shell=True)

# use bam to bed to get borders of alignments
command = 'bedtools bamtobed -i '+bam_output_file+' > '+bed_output_file
out = subprocess.check_output(command, shell=True)

Checking Lelystad.PRRSV_1, length:15111
Checking VR2332.PRRSV_2, length:15411
BAM file:/scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/WGS_Canadian_PMID32817228.sorted.bam
Mut span:1
Mutation list:
Min read:1
Max mismatch:100000000
Reference genome:/scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/PRRSV_reference_genomes.renamed.fasta
1...
Switch to VR2332.PRRSV_2, length:15411


In [30]:
mut_df = pd.read_csv(rnaseqmut_output_file,delimiter="\t",index_col=None,header=None)
mut_df.columns = ['chr','pos','ref','alt','ref_f','ref_b','alt_f','alt_b']
mut_df = pd.merge(mut_df,mut_df.groupby(['chr','pos']).agg({'alt_f':np.sum}).reset_index().rename(columns={'alt_f':'sum_alt_f'}),how='left',on=['chr','pos'])

aggr_df = mut_df.groupby(['chr','pos']).agg({'ref_f':np.max,'sum_alt_f':max,'alt_f':max}).reset_index().rename(columns={'alt_f':'max_alt_f'})
aggr_df['sum_f'] = aggr_df['ref_f']+aggr_df['sum_alt_f']
aggr_df['majAF'] = aggr_df[['ref_f','max_alt_f']].max(1)/aggr_df['sum_f']*100

bed_df = pd.read_csv(bed_output_file,delimiter="\t",index_col=None,header=None)
gr = bed_df.groupby([0]).agg({1:min,2:max}).reset_index()
a = []
for index,row in gr.iterrows():
    chromosome = row[0]
    span = list(range(row[1]+1,row[2]))
    for pos in span:
        a.append([chromosome,pos-1,pos])
bedgraph_df = pd.DataFrame(a,columns = ['chr','start','end'])
bedgraph_df = pd.merge(bedgraph_df,aggr_df[['chr','pos','majAF']].rename(columns={'pos':'end'}),how='left',on=['chr','end'])
bedgraph_df['majAF'] = bedgraph_df['majAF'].fillna(100) # if no mutations reported -> complete conservation
bedgraph_df.to_csv(bedgraph_output_file, sep=str('\t'),header=False,index=None)

# prepare bigwig displaying % of conserved nt in WGS data
command = 'bedGraphToBigWig '+bedgraph_output_file+' '+chrom_sizes_file+' '+bigwig_output_file
out = subprocess.check_output(command, shell=True)

### MSA

In [60]:
# conda activate primer_probe_design

ref_genome_title = 'prrsv_lelystad_vr2332'
github_genome_dir = subdirs['PRRSV_github_dir']+ref_genome_title+'/'

fasta_input_dir = subdirs['PRRSV_materials_dir']+'WGS_PMID32817228/'
msa_output_dir = github_genome_dir
command = 'mkdir -p '+msa_output_dir
out = subprocess.check_output(command, shell=True)

genome_file = subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.renamed.fasta'
chrom_sizes_file = github_genome_dir+ref_genome_title+'.chrom.sizes'

fasta_input_file = fasta_input_dir+'whole_genome_seqs.PRRSV_2.renamed.fasta'

anchor_chromosome = 'VR2332.PRRSV_2'

msa_input_file = fasta_input_dir+'whole_genome_seqs_and_reference.PRRSV_2.fasta'
msa_output_fasta_file = msa_output_dir+'WGS_Canadian_PMID32817228.msa.fasta'
msa_output_maf_file = msa_output_dir+'WGS_Canadian_PMID32817228.msa.maf'
msa_output_mafFrames_file = msa_output_dir+'WGS_Canadian_PMID32817228.msa.mafFrames.txt'
msa_output_mafFramesBB_file = msa_output_dir+'WGS_Canadian_PMID32817228.msa.mafFrames.bb'

msa_output_CustomMafFrames_file = msa_output_dir+'WGS_Canadian_PMID32817228.msa.Custom.mafFrames.txt'
msa_output_CustomMafFramesBB_file = msa_output_dir+'WGS_Canadian_PMID32817228.msa.Custom.mafFrames.bb'

msa_output_mafSummary_file = msa_output_dir+'WGS_Canadian_PMID32817228.msa.mafSummary.bb'

msa_output_bigMaf_file = msa_output_dir+'WGS_Canadian_PMID32817228.msa.bigMaf.txt'
msa_output_bb_file = msa_output_dir+'WGS_Canadian_PMID32817228.msa.bb'

# if we want to INCLUDE PRRSV-1 in the MSA
# command = 'cat <(seqkit grep -p "'+anchor_chromosome+'" '+genome_file+') <(seqkit grep -v -p "'+anchor_chromosome+'" '+genome_file+') '+fasta_input_file+' > '+msa_input_file+'; '
command = 'cat <(seqkit grep -p "'+anchor_chromosome+'" '+genome_file+') '+fasta_input_file+' > '+msa_input_file+'; '
command = command+'mafft '+msa_input_file+' > '+msa_output_fasta_file
print(command)

cat <(seqkit grep -p "VR2332.PRRSV_2" /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/PRRSV_reference_genomes.renamed.fasta) /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/WGS_PMID32817228/whole_genome_seqs.PRRSV_2.renamed.fasta > /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/WGS_PMID32817228/whole_genome_seqs_and_reference.PRRSV_2.fasta; mafft /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/WGS_PMID32817228/whole_genome_seqs_and_reference.PRRSV_2.fasta > /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/WGS_Canadian_PMID32817228.msa.fasta


In [61]:
# transform MSA in .fasta format to MAF format compatible with UCSC 

from Bio import AlignIO
from Bio.Align import MultipleSeqAlignment

input_handle = open(msa_output_fasta_file, "r")
alignments = AlignIO.parse(input_handle, "fasta")

output_handle = open(msa_output_maf_file+'.tmp', "w")
AlignIO.write(alignments, output_handle, "maf")

output_handle.close()
input_handle.close()

# open again, to fix the format
input_handle = open(msa_output_maf_file+'.tmp', "r")
alignments = AlignIO.parse(input_handle, "maf")
output_handle = open(msa_output_maf_file, "w")

for alignment in alignments:
    modified_seqs = []
    for k,record in enumerate(alignment):
        if k == 0:
            record.id = ref_genome_title+'.'+record.id
            defined_srcSize = len([elem for elem in list(record.seq) if elem!='-'])
        record.annotations['srcSize']=defined_srcSize
        modified_seqs.append(record)
    modified_alignment = MultipleSeqAlignment(modified_seqs)
    AlignIO.write(modified_alignment, output_handle, "maf")
    
input_handle.close()
output_handle.close()

command = 'rm '+msa_output_maf_file+'.tmp'
out = subprocess.check_output(command, shell=True)

In [62]:
# prepare bigMAF

command = 'mafToBigMaf '+ref_genome_title+' '+msa_output_maf_file+' stdout | sort -k1,1 -k2,2n > '+msa_output_bigMaf_file
out = subprocess.check_output(command, shell=True)

command = 'bedToBigBed -type=bed3+1 -as='+github_genome_dir+'bigMaf.as -tab '+msa_output_bigMaf_file+' '+chrom_sizes_file+' '+msa_output_bb_file
out = subprocess.check_output(command, shell=True)

pass1 - making usageList (1 chroms): 3 millis
pass2 - checking and writing primary data (1 records, 4 fields): 247 millis


In [63]:
# make a default mafFrames file, following the instruction from UCSC: https://genome.ucsc.edu/goldenpath/help/bigMaf.html

command = 'genePredSingleCover '+subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.genePred '+msa_output_dir+'USSPD_annotation.single.gp'
out = subprocess.check_output(command, shell=True)

command = 'genePredToMafFrames '+ref_genome_title+' '+msa_output_maf_file+' '+msa_output_mafFrames_file+' '+ref_genome_title+' '+msa_output_dir+'USSPD_annotation.single.gp'
out = subprocess.check_output(command, shell=True)

command = 'bedToBigBed -type=bed3+8 -as='+msa_output_dir+'mafFrames.as -tab '+msa_output_mafFrames_file+' '+chrom_sizes_file+' '+msa_output_mafFramesBB_file
out = subprocess.check_output(command, shell=True)

pass1 - making usageList (1 chroms): 0 millis
pass2 - checking and writing primary data (3 records, 11 fields): 2 millis


In [64]:
# make custom mafFrames file, to include all annotated protein-coding regions, in ALL MSA entities!
# use USS annotation because it's more comprehensive

gp_df = pd.read_csv(subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.USSPD_annotation.genePred',delimiter="\t",index_col=None,header=None)
gp_df = gp_df.sort_values([1,3,4]).reset_index(drop=True)


gp_bed = gp_df.copy()
gp_bed = gp_bed[[1,0,5,6,14,2]].rename(columns={1:0,0:3,5:1,6:2,14:4,2:5})[range(0,6)]

a = []
for strand in ['+','-']:
    if len(gp_bed.loc[gp_bed[5]==strand])>0:
        gp_bed.loc[gp_bed[5]==strand].to_csv(subdirs['temp_dir']+'gp.'+strand+'.unsorted.bed', sep=str('\t'),header=False,index=None)
        command = 'sort-bed '+subdirs['temp_dir']+'gp.'+strand+'.unsorted.bed'+' > '+subdirs['temp_dir']+'gp.'+strand+'.sorted.bed'
        out = subprocess.check_output(command, shell=True)
        command = 'bedops --partition '+subdirs['temp_dir']+'gp.'+strand+'.sorted.bed'+' | sort -k1,1 -k2,2n > '+subdirs['temp_dir']+'gp.'+strand+'.partitioned.bed'
        out = subprocess.check_output(command, shell=True)
        command = 'bedtools intersect -wa -wb -sorted -a '+subdirs['temp_dir']+'gp.'+strand+'.sorted.bed'+' -b '+subdirs['temp_dir']+'gp.'+strand+'.partitioned.bed > '+subdirs['temp_dir']+'gp.'+strand+'.intersection.bed'
        out = subprocess.check_output(command, shell=True)
        intersection_df = pd.read_csv(subdirs['temp_dir']+'gp.'+strand+'.intersection.bed',delimiter="\t",index_col=None,header=None)
    
        if strand == '+':
            intersection_df['frame_at_cutPos'] = (abs(intersection_df[7]-intersection_df[1]))%3
        else:
            intersection_df['frame_at_cutPos'] = (abs(intersection_df[8]-intersection_df[2]))%3
        
        intersection_df['gene_name'] = intersection_df[3].str.split(' ',expand=True)[0]
        intersection_df['gene_names'] = intersection_df.groupby([6,7,8])['gene_name'].transform(lambda x: '__'.join(x))
        partitions_frames_df = intersection_df.groupby([6,7,8,'gene_names']).agg({'frame_at_cutPos':{max,min}}).reset_index()
        partitions_frames_df.columns = ['chr','start','end','gene_names','frame_max','frame_min']
        
        sel_CDS_partitions = partitions_frames_df.loc[partitions_frames_df['frame_max']==partitions_frames_df['frame_min']].reset_index(drop=True) # require the same
        sel_CDS_partitions['consensus_frame'] = sel_CDS_partitions['frame_max']
        sel_CDS_partitions['strand'] = strand
        a.append(sel_CDS_partitions)

sel_CDS_partitions = pd.concat(a).reset_index(drop=True)

# we put the same ORFs for all records in MAF
input_handle = open(msa_output_maf_file, "r")
alignments = AlignIO.parse(input_handle, "maf")

maf_record_ids = []
for alignment in alignments:
    for k,record in enumerate(alignment):
        maf_record_ids.append(record.id)    
input_handle.close()

a = []
for maf_record in maf_record_ids:
    sel_CDS_partitions['maf_record'] = maf_record
    a.append(sel_CDS_partitions.copy())
custom_mafFrames_df = pd.concat(a).reset_index(drop=True)

custom_mafFrames_df[0] = custom_mafFrames_df['chr']
custom_mafFrames_df[1] = custom_mafFrames_df['start']
custom_mafFrames_df[2] = custom_mafFrames_df['end']
custom_mafFrames_df[3] = custom_mafFrames_df['maf_record']
custom_mafFrames_df[4] = custom_mafFrames_df['consensus_frame']
custom_mafFrames_df[5] = custom_mafFrames_df['strand']
custom_mafFrames_df[6] = custom_mafFrames_df['gene_names']
custom_mafFrames_df[7] = -1
custom_mafFrames_df[8] = -1
custom_mafFrames_df[9] = 1
custom_mafFrames_df[10] = 1
custom_mafFrames_df[range(0,11)].to_csv(msa_output_CustomMafFrames_file, sep=str('\t'),header=False,index=None)

command = 'bedToBigBed -sort -type=bed3+8 -as='+msa_output_dir+'mafFrames.as -tab '+msa_output_CustomMafFrames_file+' '+chrom_sizes_file+' '+msa_output_CustomMafFramesBB_file
out = subprocess.check_output(command, shell=True)

pass1 - making usageList (1 chroms): 1 millis
pass2 - checking and writing primary data (1767 records, 11 fields): 6 millis


In [65]:
# we downloaded compiled hgLoadMafSummary directly from UCSC, it seems to be unavailable from BIOCONDA

# conda activate py312

command = 'cd '+msa_output_dir+' && hgLoadMafSummary -minSeqSize=1 -test '+ref_genome_title+' bigMafSummary '+msa_output_maf_file+' && '+\
'cut -f2- bigMafSummary.tab | sort -k1,1 -k2,2n > bigMafSummary.bed && '+\
'bedToBigBed -type=bed3+4 -as=mafSummary.as -tab bigMafSummary.bed '+chrom_sizes_file+' '+msa_output_mafSummary_file
out = subprocess.check_output(command, shell=True)
print(command)

cd /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/ && hgLoadMafSummary -minSeqSize=1 -test prrsv_lelystad_vr2332 bigMafSummary /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/WGS_Canadian_PMID32817228.msa.maf && cut -f2- bigMafSummary.tab | sort -k1,1 -k2,2n > bigMafSummary.bed && bedToBigBed -type=bed3+4 -as=mafSummary.as -tab bigMafSummary.bed /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/prrsv_lelystad_vr2332.chrom.sizes /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/WGS_Canadian_PMID32817228.msa.mafSummary.bb


Indexing and tabulating /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/WGS_Canadian_PMID32817228.msa.maf
Created 92 summary blocks from 92 components and 1 mafs from /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/WGS_Canadian_PMID32817228.msa.maf
pass1 - making usageList (1 chroms): 0 millis
pass2 - checking and writing primary data (92 records, 7 fields): 1 millis


In [ ]:
###
# do phylogenetic analysis - visualize subtypes
###



## Primer alignment

In [3]:
###
# newly designed primers
###

ref_genome_title = 'prrsv_lelystad_vr2332'
github_genome_dir = subdirs['PRRSV_github_dir']+ref_genome_title+'/'

fasta_input_dir = subdirs['PRRSV_materials_dir']+'WGS_PMID32817228/'
msa_output_dir = github_genome_dir

varvamp_single_dir = msa_output_dir+'varvamp/single/'
varvamp_qPCR_dir = msa_output_dir+'varvamp/qPCR/'

varvamp_single_primers_df = pd.read_csv(varvamp_single_dir+'primers.bed',delimiter="\t",index_col=None,header=None)
varvamp_qPCR_primers_df = pd.read_csv(varvamp_qPCR_dir+'primers.bed',delimiter="\t",index_col=None,header=None)

varvamp_single_primers_df['ID'] = 'varVAMP_single_PRRSV2_'+varvamp_single_primers_df[3].str.replace('_LEFT','').str.replace('_RIGHT','')
varvamp_single_primers_df['side'] = varvamp_single_primers_df.apply(lambda x:('FW' if x[3].endswith('_LEFT') else 'REV'),1)
varvamp_tmp1_df = pd.merge(varvamp_single_primers_df.loc[varvamp_single_primers_df['side']=='FW'][['ID',6]].rename(columns={6:'FW'}),
        varvamp_single_primers_df.loc[varvamp_single_primers_df['side']=='REV'][['ID',6]].rename(columns={6:'REV'}),how='inner',on='ID')

varvamp_qPCR_primers_df['ID'] = 'varVAMP_qPCR_PRRSV2_'+varvamp_qPCR_primers_df[3].str.replace('_LEFT','').str.replace('_RIGHT','').str.replace('_PROBE','')
varvamp_tmp2_df = pd.merge(varvamp_qPCR_primers_df.loc[varvamp_qPCR_primers_df[3].str.endswith('_LEFT')][['ID',6]].rename(columns={6:'FW'}),
        varvamp_qPCR_primers_df.loc[varvamp_qPCR_primers_df[3].str.endswith('_RIGHT')][['ID',6]].rename(columns={6:'REV'}),how='inner',on='ID')
varvamp_tmp2_df = pd.merge(varvamp_tmp2_df,
        varvamp_qPCR_primers_df.loc[varvamp_qPCR_primers_df[3].str.endswith('_PROBE')][['ID',6]].rename(columns={6:'PROBE'}),how='inner',on='ID')

varvamp_primers_df = pd.concat([varvamp_tmp1_df,varvamp_tmp2_df]).reset_index(drop=True)
varvamp_primers_df['PROBE'] = varvamp_primers_df['PROBE'].fillna('XXX')
varvamp_primers_df['targeted_locus'] = 'unknown'

# primers from literature
primers_df = pd.read_csv(subdirs['PRRSV_materials_dir']+'PRRSV_primers.tsv',delimiter="\t",index_col=None,header=0)

# primers_df = pd.concat([primers_df[['ID','FW','REV','PROBE','targeted_locus']],
#             varvamp_primers_df[['ID','FW','REV','PROBE','targeted_locus']]]).reset_index(drop=True) # union of existing and newly designed primers
primers_df = primers_df[['ID','FW','REV','PROBE','targeted_locus']]

In [12]:
# for student projects
primers_df = primers_df.loc[primers_df['ID'].str.contains('|'.join(['PRRSV2','PRRSV1']))].reset_index(drop=True)
primers_df['target'] = primers_df.apply(lambda x:('PRRSV1' if 'PRRSV1' in x['ID'] else 'PRRSV2'),1)

In [13]:
primers_df

,ID,FW,REV,PROBE,targeted_locus,target
0,Wernike2012_for_qRT_PCR_PRRSV2,ATRATGRGCTGGCATTC,ACACGGTCGCCCTAATTG,TGTGGTGAATGGCACTGATTGACA,ORF7_3primeUTR,PRRSV2
1,Wernike2012_for_qRT_PCR_PRRSV1_EU1,GCACCACCTCACCCRRAC,CAGTTCCTGCRCCYTGAT,CCTCTGYYTGCAATCGATCCAGAC,ORF6_ORF7,PRRSV1
2,Wernike2012_for_qRT_PCR_PRRSV1_EU2,CAGATGCAGAYTGTGTTGCCT,TGGAGDCCTGCAGCACTTTC,ATACATTCTGGCCCCTGCCCAYCACGT,ORF6_ORF7,PRRSV1
3,Balka2018_for_NGS_PRRSV1_option1,TTCACAGATTATGTGGCCCATGTGAC,CGYGACACCTTRAGGGCRTATATCAT,XXX,ORF5,PRRSV1
4,Balka2018_for_NGS_PRRSV1_option2,GCGTYACRGATTATGTGGCYCAYGT,CGYGACACCTTRAGGGCRTATATCAT,XXX,ORF5,PRRSV1
5,Balka2018_for_NGS_PRRSV2,GTCAAYTTTACCAGYTAYGTCCAACA,AGRGCATATATCATYACYGGCGTGTA,XXX,ORF5,PRRSV2


In [15]:
# list all the different nt codes we have in the existing primers
l = list(primers_df['FW'].unique())+list(primers_df['REV'].unique())+list(primers_df['PROBE'].unique())
l = list(np.unique(l))
a = []
for elem in l:
    a = a+list(np.unique(list(elem)))
a = list(np.unique(a))
', '.join([str(elem) for elem in a if elem not in ['A','C','G','T']])

'D, R, X, Y'

In [20]:
###
# prepare paired fasta files for primer sequences
###

# IUPAC codes
nt_dict = {'D':['A', 'G', 'T'],'H':['A','C','T'],'V':['A','C','G'],'B':['C','G','T'],
           'M':['A','C'],'K':['G','T'],'S':['C','G'],'W':['A','T'],'R':['A','G'],'Y':['C','T'],
           'X':['A','C','G','T'],'N':['A','C','G','T']}

def create_fasta_files(seq_elements):

    a = []
    for index,row in primers_df.iterrows():
        new_row = row.copy()
        seqs_dict = {}
        for seq_element in seq_elements:
            if row[seq_element]!='XXX':
                new_seqs = []
                for elem in list(row[seq_element]):
                    new_nts = (nt_dict[elem] if elem in nt_dict.keys() else [elem])
                    if len(new_seqs)==0:
                        new_seqs = new_nts
                    else:
                        # update new_seqs
                        new_seqs_update = []
                        for new_seq in new_seqs:
                            for new_nt in new_nts:
                                new_seqs_update.append(new_seq+new_nt)
                        new_seqs = new_seqs_update
            else:
                new_seqs = [row[seq_element]]
            seqs_dict[seq_element] = new_seqs    
        new_row_df = pd.DataFrame(new_row).transpose()
        new_row_df = new_row_df.drop(seq_elements,axis=1)
        new_row_df['t']=1
    
        # make cartesian product of all possible primer variations
        for k,elem in enumerate(seqs_dict):
            tmp = pd.DataFrame(seqs_dict[elem],columns=[elem])
            tmp['t']=1
            if k==0:
                cartesian = tmp.copy()
            else:
                cartesian = pd.merge(cartesian,tmp,how='inner',on=['t'])
    
        new_row_df = pd.merge(new_row_df,cartesian,how='inner',on=['t'])
        a.append(new_row_df)
    full_primers_df = pd.concat(a).reset_index(drop=True)
    
    # make unique indices for each row
    full_primers_df['temp_index'] = full_primers_df.index
    full_primers_df['within_group_index'] = full_primers_df.groupby('ID')['temp_index'].rank(method='average',ascending=True).astype('int')
    # full_primers_df = pd.merge(full_primers_df,primers_df[['ID','target']],how='left',on='ID')
    full_primers_df['fasta_ID'] = full_primers_df['target']+'_'+full_primers_df['ID'].str.replace('_','.')+'.'+full_primers_df['within_group_index'].astype('str').str.replace('_','.')+'.'+full_primers_df['targeted_locus'].str.replace('_','.')
    
    output_dir = subdirs['PRRSV_materials_dir']+'primer_sets/existing_primers_for_students/'
    command = 'mkdir -p '+output_dir
    out = subprocess.check_output(command, shell=True)
    
    fasta_files = []
    for k,seq_element in enumerate(seq_elements):
        fasta_files.append(open(output_dir+seq_element+'.fasta', "w"))
    # also write primers to a common .fasta file
    common_fasta_file = open(output_dir+'primers.fasta', "w")
    for index,row in full_primers_df.iterrows():
        common_fasta_element = '>'+row['fasta_ID']+'\n'
        for k,seq_element in enumerate(seq_elements):
            if row[seq_element]!='XXX':
                fasta_element = common_fasta_element+row[seq_element]+'\n'
                fasta_files[k].write(fasta_element)
                fasta_element = '>'+row['fasta_ID']+'_'+seq_element+'\n'+row[seq_element]+'\n'
                common_fasta_file.write(fasta_element)
    
    for k,seq_element in enumerate(seq_elements):
        fasta_files[k].close()
    common_fasta_file.close()

In [21]:
create_fasta_files(['FW','REV'])
# create_fasta_files(['PROBE'])

In [35]:
# Try STAR

### STAR Index

ref_genome_title = 'prrsv_lelystad_vr2332'
github_genome_dir = subdirs['PRRSV_github_dir']+ref_genome_title+'/'
genome_file = subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.renamed.fasta'
STAR_reference_dir = subdirs['PRRSV_reference_genomes_dir']+'STAR/'

script_path = subdirs['scripts_dir']+'create_star_index_PRRSV.sbatch'

f = open(script_path, "w")
preambula = \
"""#!/bin/bash
  
#SBATCH --job-name=create_star_index_PRRSV
#SBATCH --time=5:00:00
#SBATCH --qos=6hours
#SBATCH --output="""+subdirs['slurm_dir']+"""%A_%a.out
#SBATCH --error="""+subdirs['slurm_dir']+"""%A_%a.err
#SBATCH --cpus-per-task=12
#SBATCH --mem=20G

source ~/.bashrc
conda activate primer_probe_design
"""
f.write(preambula+'\n')
command = """
STAR --runThreadN 12 \
--runMode genomeGenerate \
--genomeSAindexNbases 6 \
--genomeDir """+STAR_reference_dir+""" \
--genomeFastaFiles """+genome_file
f.write(command+'\n')

f.close()
print(command)
print('sbatch '+script_path)


STAR --runThreadN 12 --runMode genomeGenerate --genomeSAindexNbases 6 --genomeDir /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/STAR/ --genomeFastaFiles /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/PRRSV_reference_genomes.renamed.fasta
sbatch /scicore/home/zavolan/GROUP/Primer_Probe_design/scripts/create_star_index_PRRSV.sbatch


In [109]:
ref_genome_title = 'prrsv_lelystad_vr2332'
github_genome_dir = subdirs['PRRSV_github_dir']+ref_genome_title+'/'
genome_file = subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.renamed.fasta'
STAR_reference_dir = subdirs['PRRSV_reference_genomes_dir']+'STAR/'

primers_fasta_dir = subdirs['PRRSV_materials_dir']+'primer_sets/existing_primers_and_varVAMP/'
script_path = subdirs['scripts_dir']+'map_primers_to_PRRSV_reference_genome_with_STAR.sbatch'

cpus = 4

f = open(script_path, "w")
preambula = \
"""#!/bin/bash
  
#SBATCH --job-name=map_primers_to_PRRSV_reference_genome_with_STAR
#SBATCH --time=5:00:00
#SBATCH --qos=6hours
#SBATCH --output="""+subdirs['slurm_dir']+"""%A_%a.out
#SBATCH --error="""+subdirs['slurm_dir']+"""%A_%a.err
#SBATCH --cpus-per-task="""+str(cpus)+"""
#SBATCH --mem=20G

source ~/.bashrc
conda activate primer_probe_design
"""
f.write(preambula+'\n')

query_fasta_files = [primers_fasta_dir+'FW.fasta '+primers_fasta_dir+'REV.fasta',primers_fasta_dir+'PROBE.fasta']
out_prefices = [primers_fasta_dir+'primers.',primers_fasta_dir+'PROBE.']

for k,query_fasta_file in enumerate(query_fasta_files):
    command = """STAR \
--runMode alignReads \
--alignEndsType Local \
--runThreadN """+str(cpus)+""" \
--genomeLoad NoSharedMemory \
--outSAMtype BAM SortedByCoordinate \
--genomeDir """+STAR_reference_dir+""" \
--readFilesIn """+query_fasta_file+""" \
--outFileNamePrefix """+out_prefices[k]+""" \
--outFilterType Normal \
--outSJfilterOverhangMin 100 100 100 100 \
--alignSJDBoverhangMin 100 \
--alignSJoverhangMin 100 \
--outFilterMismatchNmax 999 \
--outFilterMismatchNoverReadLmax 0.2 \
--outFilterScoreMinOverLread 0.3 \
--outFilterMatchNminOverLread 0.3 \
--twopassMode None \
--outBAMsortingThreadN """+str(cpus)+""" \
--alignTranscriptsPerReadNmax 50000 \
--winAnchorMultimapNmax 50000 \
--outFilterMultimapNmax 50000 \
--outSAMprimaryFlag AllBestScore \
--outSAMattributes NH HI AS nM MD; \
cp -f """+out_prefices[k]+'Aligned.sortedByCoord.out.bam '+github_genome_dir+"""; \
samtools index """+out_prefices[k]+'Aligned.sortedByCoord.out.bam'+"""; \
cp -f """+out_prefices[k]+'Aligned.sortedByCoord.out.bam.bai '+github_genome_dir+"""; \
"""
    print(command)
    f.write(command+'\n')
f.close()
print('sbatch '+script_path)

STAR --runMode alignReads --alignEndsType Local --runThreadN 4 --genomeLoad NoSharedMemory --outSAMtype BAM SortedByCoordinate --genomeDir /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/reference_genomes/STAR/ --readFilesIn /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/primer_sets/existing_primers_and_varVAMP/FW.fasta /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/primer_sets/existing_primers_and_varVAMP/REV.fasta --outFileNamePrefix /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/materials/primer_sets/existing_primers_and_varVAMP/primers. --outFilterType Normal --outSJfilterOverhangMin 100 100 100 100 --alignSJDBoverhangMin 100 --alignSJoverhangMin 100 --outFilterMismatchNmax 999 --outFilterMismatchNoverReadLmax 0.2 --outFilterScoreMinOverLread 0.3 --outFilterMatchNminOverLread 0.3 --twopassMode None --outBAMsortingThreadN 4 --alignTranscriptsPerReadNmax 50000 --winAnchorMultimapNmax 50000 --outFilterMultimapNmax 50000 --outSAMprimaryFla

In [ ]:
# Try RIblast



## Primer search

In [73]:
# conda activate primer_probe_design

ref_genome_title = 'prrsv_lelystad_vr2332'
github_genome_dir = subdirs['PRRSV_github_dir']+ref_genome_title+'/'

fasta_input_dir = subdirs['PRRSV_materials_dir']+'WGS_PMID32817228/'
msa_output_dir = github_genome_dir
varvamp_single_dir = msa_output_dir+'varvamp/single/'
varvamp_qPCR_dir = msa_output_dir+'varvamp/qPCR/'

command = 'mkdir -p '+msa_output_dir+' '+varvamp_single_dir+' '+varvamp_qPCR_dir
out = subprocess.check_output(command, shell=True)

genome_file = subdirs['PRRSV_reference_genomes_dir']+'PRRSV_reference_genomes.renamed.fasta'
chrom_sizes_file = github_genome_dir+ref_genome_title+'.chrom.sizes'

fasta_input_file = fasta_input_dir+'whole_genome_seqs.PRRSV_2.renamed.fasta'

anchor_chromosome = 'VR2332.PRRSV_2'
msa_fasta_file = msa_output_dir+'WGS_Canadian_PMID32817228.msa.fasta'
msa_maf_file = msa_output_dir+'WGS_Canadian_PMID32817228.msa.maf'

command = 'varvamp single '+msa_fasta_file+' '+varvamp_single_dir+'; '
command = command+'varvamp qpcr '+msa_fasta_file+' '+varvamp_qPCR_dir+'; '

print(command)

varvamp single /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/WGS_Canadian_PMID32817228.msa.fasta /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/varvamp/single/; varvamp qpcr /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/WGS_Canadian_PMID32817228.msa.fasta /scicore/home/zavolan/GROUP/Primer_Probe_design/PRRSV/github/prrsv_lelystad_vr2332/varvamp/qPCR/; 


## make hub

In [110]:
ref_genome_title = 'prrsv_lelystad_vr2332'
github_genome_dir = subdirs['PRRSV_github_dir']+ref_genome_title+'/'
command = 'mkdir -p '+github_genome_dir
out = subprocess.check_output(command, shell=True)

# we need to extract necessary features for WGS data
from Bio import AlignIO
from Bio.Align import MultipleSeqAlignment

input_handle = open(github_genome_dir+'WGS_Canadian_PMID32817228.msa.maf', "r")
alignments = AlignIO.parse(input_handle, "maf")

a = []
for alignment in alignments:
    for k,record in enumerate(alignment):
        if k!=0:
            a.append(record.id)    
input_handle.close()

WGS_seqs_order = ' '.join(a)
WGS_seqs_labels = []
for k,elem in enumerate(a):
    label = elem.lower().replace('.','_')
    WGS_seqs_labels.append(elem+'='+label)
WGS_seqs_labels = ' '.join(WGS_seqs_labels)

In [69]:
# define hub.txt
hub_descr = """hub PRRSV_Hub
shortLabel PRRSV
longLabel PRRSV
genomesFile genomes.txt
email magmir71@gmail.com
descriptionUrl https://raw.githubusercontent.com/zavolanlab/primer_probe_design/main/track_descriptions/Track_hub_description.html"""
f = open(subdirs['PRRSV_github_dir']+'hub.txt', "w")
f.write(hub_descr)
f.close()

genomes_file_txt = """genome """+ref_genome_title+"""
trackDb """+ref_genome_title+"""/trackDb.txt
groups groups.txt
description genome sequences of PRRSV reference strains Lelystad (PRRSV type-1) and VR2332 (PRRSV type-2)
twoBitPath """+ref_genome_title+"""/"""+ref_genome_title+""".2bit
organism Betaarterivirus americense
defaultPos VR2332.PRRSV_2:1-1000
orderKey 4800
scientificName Betaarterivirus americense
htmlPath """+ref_genome_title+"""/description.html"""
f = open(subdirs['PRRSV_github_dir']+'genomes.txt', "w")
f.write(genomes_file_txt)
f.close()

# define trackDb file
trackDb_file = """track cytoBandIdeo
longLabel Chromosome ideogram with cytogenetic bands
shortLabel cytoBandIdeo
bigDataUrl cytoBandIdeo.bigBed
type bigBed
#For more info. on the cytoBandIdeo track see http://genomewiki.ucsc.edu/index.php/Assembly_Hubs#Cytoband_Track

track genebank_features
longLabel Genebank genome features 
shortLabel GB features
type bigGenePred
description Genome features annotated in the Genebank files of the reference genomes
visibility pack
bigDataUrl """+ref_genome_title+""".gene_annotation.bb
baseColorDefault genomicCodons
group genes
html genes

track USSPD_features
longLabel US swine pathogene database features 
shortLabel USSPD features
type bigGenePred
description Feautures were obtained from USSPD annotations of curated GB sequences, mapped to the reference strains
visibility pack
bigDataUrl """+ref_genome_title+""".USSPD_annotation.bb
baseColorDefault genomicCodons
group genes
html genes

track wgs_canadian_PA
longLabel pairwise alignment, Canadian WGS, PMID 32817228
shortLabel wgs_canadian_PA
type bam
db """+ref_genome_title+"""
description Pairwise alignments of assembled WGS genomes to reference strains
maxWindowToDraw 200000
visibility hide
bigDataUrl WGS_Canadian_PMID32817228.sorted.bam
group wgs
html WGS_Canadian

track wgs_canadian_cons_PA
longLabel nt SNV conservation, %, Canadian WGS, PMID 32817228, based on PA
shortLabel wgs_canadian_cons_PA
type bigWig
db """+ref_genome_title+"""
description nt SNV conservation %, based on pairwise alignments of assembled WGS genomes to reference strains
maxWindowToQuery 2000
alwaysZero on
viewLimits 0:100
viewLimitsMax 0:100
windowingFunction mean+whiskers
gridDefault on
color 20,72,156
maxHeightPixels 100:50:8
visibility hide
bigDataUrl WGS_Canadian_PMID32817228.bigwig
group wgs
html WGS_Canadian

track wgs_canadian_msa
longLabel multiple sequence alignment, Canadian WGS, PMID 32817228 
shortLabel wgs_canadian_msa
type bigMaf
db """+ref_genome_title+"""
description multiple sequence alignment, with anchoring to VR2332.PRRSV_2
visibility pack
bigDataUrl WGS_Canadian_PMID32817228.msa.bb
summary WGS_Canadian_PMID32817228.msa.mafSummary.bb
frames WGS_Canadian_PMID32817228.msa.Custom.mafFrames.bb
speciesOrder """+WGS_seqs_order+"""
speciesLabels """+WGS_seqs_labels+"""
group wgs
html WGS_Canadian

track existing_primers
longLabel Established primers referenced in WOAH Terrestrial Manual 2021  
shortLabel existing_primers
type bam
db """+ref_genome_title+"""
description Pairwise alignments of established primers to reference strains
maxWindowToDraw 200000
visibility pack
pairEndsByName .
pairSearchRange 10000
showNames on
bigDataUrl primers.Aligned.sortedByCoord.out.bam
group primers_and_probes
html WGS_Canadian

track existing_probes
longLabel Established probes referenced in WOAH Terrestrial Manual 2021  
shortLabel existing_probes
type bam
db """+ref_genome_title+"""
description Pairwise alignments of established probes to reference strains
maxWindowToDraw 200000
visibility pack
showNames on
bigDataUrl PROBE.Aligned.sortedByCoord.out.bam
group primers_and_probes
html WGS_Canadian
"""
f = open(subdirs['PRRSV_github_dir']+ref_genome_title+'/trackDb.txt', "w")
f.write(trackDb_file)
f.close()

# define groups file
group_txt_file = """name user
label Custom
priority 1
defaultIsClosed 1

name map
label Assembly
priority 2
defaultIsClosed 0

name genes
label Genes
priority 3
defaultIsClosed 0

name wgs
label WGS
priority 4
defaultIsClosed 0

name primers_and_probes
label Primers and Probes
priority 5
defaultIsClosed 0
"""
f = open(subdirs['PRRSV_github_dir']+'groups.txt', "w")
f.write(group_txt_file)
f.close()

## Drafts

In [375]:
cur_dir = '/scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/'

organism = 'human'
organism_github_subfolder = 'GRCh38'
github_branch = 'dev'
chrom_sizes_file = file_paths[organism+'_chrom_sizes']

github_dir = subdirs[organism+'_github_dir']
out_dir = github_dir+'total/'
tmp_dir = cur_dir+'tmp/'
trackhub_chunks_dir = tmp_dir+organism+'/'
big_data_dir = subdirs['shared_project_dir']+(organism if organism!='celegans' else 'c_elegans')+'/polyAsite_Atlas_3/for_trackhub/'

out = subprocess.check_output('mkdir -p '+tmp_dir +' '+out_dir+' '+trackhub_chunks_dir+' '+big_data_dir, shell=True)

SCINPAS_tsv = pd.read_csv(file_paths[organism+'_output_PAS_tsv_gz'],delimiter="\t",index_col=None,header=0,compression='gzip')

In [4]:
# SCINPAS_tsv = SCINPAS_tsv.loc[SCINPAS_tsv['chrom']=='chr3'].reset_index(drop=True) # temporary filtration, for testing

In [377]:
len(SCINPAS_tsv)

18432135

In [378]:
# renamed tissues!

tissues = ['brain', 'bloodImmune','eye', 'intervertebral disc', 'nasal mucosa', 'skin', 'corpus cavernosum',  'heart', 'breast',
       'lung', 'tracheal epithelium', 'pancreas','intestine','kidney parenchyma', 'liver', 'uterus', 'ureter','prostate']

In [24]:
# make bigbeds for annotation and bigwigs with average RPM, and tissue-specific RPM values

strands = {'+':'plus','-':'minus'}

command = ''
stringency_groups = {'20-21':[20, 21],'22-29':[23, 26, 29],'30-100':[32, 36, 39, 47, 51, 57, 62, 69, 77, 87, 98]}
for stringency_group in stringency_groups:
    l = stringency_groups[stringency_group]
    for strand in strands:
        SCINPAS_tsv_cur = SCINPAS_tsv.loc[(SCINPAS_tsv['stringency_level'].isin(l))&(SCINPAS_tsv['strand']==strand)].reset_index(drop=True)
        
        ### bigbed - average
        SCINPAS_tsv_cur[['chrom','chromStart','chromEnd','name','stringency_level','strand']].to_csv(out_dir+'PAS.'+str(stringency_group)+'.'+strand+'.total.bed',sep=str('\t'),header=False,index=None,quoting=csv.QUOTE_NONE)
        command = command+'sort -k1,1 -k2,2n '+out_dir+'PAS.'+str(stringency_group)+'.'+strand+'.total.bed'+' > '+out_dir+'PAS.'+str(stringency_group)+'.'+strand+'.total.sorted.bed && '
        command = command+'bedToBigBed -type=bed6 '+out_dir+'PAS.'+str(stringency_group)+'.'+strand+'.total.sorted.bed '+chrom_sizes_file+' '+out_dir+'PAS.'+str(stringency_group)+'.'+strand+'.total.bb && '
        command = command+'rm '+out_dir+'PAS.'+str(stringency_group)+'.'+strand+'.total.bed '+out_dir+'PAS.'+str(stringency_group)+'.'+strand+'.total.sorted.bed && '
        
        ### bigwig - average
        strand_long_label = strands[strand]
        SCINPAS_tsv_cur[['chrom','chromStart','chromEnd','avg_expression']].to_csv(tmp_dir+'PAS.'+str(stringency_group)+'.'+strand_long_label+'.bedgraph',sep=str('\t'),header=False,index=None,quoting=csv.QUOTE_NONE)
        command = command+'sort -k1,1 -k2,2n '+tmp_dir+'PAS.'+str(stringency_group)+'.'+strand_long_label+'.bedgraph > '+tmp_dir+'PAS.'+str(stringency_group)+'.'+strand_long_label+'.sorted.bedgraph && '
        command = command+'bedGraphToBigWig '+tmp_dir+'PAS.'+str(stringency_group)+'.'+strand_long_label+'.sorted.bedgraph '+chrom_sizes_file+' '+out_dir+'PAS.'+str(stringency_group)+'.'+strand_long_label+'.bw && '
        command = command+'rm '+tmp_dir+'PAS.'+str(stringency_group)+'.'+strand_long_label+'.bedgraph '+tmp_dir+'PAS.'+str(stringency_group)+'.'+strand_long_label+'.sorted.bedgraph && '

command

'sort -k1,1 -k2,2n /scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/github/polyAsite_Atlas_3/GRCh38/total/PAS.20-21.+.total.bed > /scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/github/polyAsite_Atlas_3/GRCh38/total/PAS.20-21.+.total.sorted.bed && bedToBigBed -type=bed6 /scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/github/polyAsite_Atlas_3/GRCh38/total/PAS.20-21.+.total.sorted.bed /scicore/home/zavolan/GROUP/Genomes/homo_sapiens/hg38.chrom.sizes /scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/github/polyAsite_Atlas_3/GRCh38/total/PAS.20-21.+.total.bb && rm /scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/github/polyAsite_Atlas_3/GRCh38/total/PAS.20-21.+.total.bed /scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/github/polyAsite_Atlas_3/GRCh38/total/PAS.20-21.+.total.sorted.bed && sort -k1,1 -k2,2n /scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/tmp/PAS.20-21.plus.bedgraph > /scicore/home/zavolan/mirono0000/Projects/SC

In [382]:
### bigwig - tissue-specific

strands = {'+':'plus','-':'minus'}

test_out_dir = github_dir+'over_tissues/'
out = subprocess.check_output('mkdir -p '+test_out_dir, shell=True)

command = ''
for strand in strands:
    strand_long_label = strands[strand]
    # SCINPAS_tsv_cur = SCINPAS_tsv.loc[(SCINPAS_tsv['strand']==strand)].reset_index(drop=True)
    for tissue in tissues:
        outfile = test_out_dir+'PAS.'+tissue+'.'+strand_long_label+'.bw'
        if not((os.path.isfile(outfile) and (os.stat(outfile).st_size > 0))):
        # SCINPAS_tsv_cur[['chrom','chromStart','chromEnd',tissue]].to_csv(test_out_dir+'PAS.'+tissue+'.'+strand_long_label+'.bedgraph',sep=str('\t'),header=False,index=None,quoting=csv.QUOTE_NONE)
            command = command+'sort -k1,1 -k2,2n "'+test_out_dir+'PAS.'+tissue+'.'+strand_long_label+'.bedgraph" > "'+test_out_dir+'PAS.'+tissue+'.'+strand_long_label+'.sorted.bedgraph" && '
            command = command+'bedGraphToBigWig "'+test_out_dir+'PAS.'+tissue+'.'+strand_long_label+'.sorted.bedgraph" '+chrom_sizes_file+' "'+test_out_dir+'PAS.'+tissue+'.'+strand_long_label+'.bw" && '
            command = command+'rm "'+test_out_dir+'PAS.'+tissue+'.'+strand_long_label+'.bedgraph" "'+test_out_dir+'PAS.'+tissue+'.'+strand_long_label+'.sorted.bedgraph" && '
command

'sort -k1,1 -k2,2n "/scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/github/polyAsite_Atlas_3/GRCh38/over_tissues/PAS.intervertebral disc.plus.bedgraph" > "/scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/github/polyAsite_Atlas_3/GRCh38/over_tissues/PAS.intervertebral disc.plus.sorted.bedgraph" && bedGraphToBigWig "/scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/github/polyAsite_Atlas_3/GRCh38/over_tissues/PAS.intervertebral disc.plus.sorted.bedgraph" /scicore/home/zavolan/GROUP/Genomes/homo_sapiens/hg38.chrom.sizes "/scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/github/polyAsite_Atlas_3/GRCh38/over_tissues/PAS.intervertebral disc.plus.bw" && rm "/scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/github/polyAsite_Atlas_3/GRCh38/over_tissues/PAS.intervertebral disc.plus.bedgraph" "/scicore/home/zavolan/mirono0000/Projects/SCINPAS_catalog/github/polyAsite_Atlas_3/GRCh38/over_tissues/PAS.intervertebral disc.plus.sorted.bedgraph" && sort -k1,1 

In [388]:
# create text chunks for trackhub configuration

max_average = str(int(SCINPAS_tsv['avg_expression'].max()))
max_tissue_level = str(int(SCINPAS_tsv[tissues].max(1).max()))
stringency_groups = {'20-21':[20, 21],'22-29':[23, 26, 29],'30-100':[32, 36, 39, 47, 51, 57, 62, 69, 77, 87, 98]}

strands = {'+':'plus','-':'minus'}
f = open(trackhub_chunks_dir+'hub_total.'+organism+'.txt', "w")

command = ''

bigbed_composite = """track PAS_catalogue
compositeTrack on
allButtonPair on
visibility squish
shortLabel PAS catalogue
longLabel 1. PAS - full catalogue
type bigBed 6 .
spectrum on
scoreMax 100
scoreMin 20
html https://raw.githubusercontent.com/zavolanlab/polyAsite_Atlas_3/"""+github_branch+"""/"""+organism_github_subfolder+"""/track_descriptions/PAS_catalogue.html

"""
f.write(bigbed_composite)

i = 0
for stringency_group in stringency_groups:
    for strand in strands:
        bigbed = """track PAS_catalogue_"""+str(stringency_group)+'_'+strand+""" 
parent PAS_catalogue on
shortLabel """+str(stringency_group)+"""% motif presence, """+strand+""" strand
longLabel """+('1. PAS - full catalogue' if i==0 else '')+"""
bigDataUrl https://raw.githubusercontent.com/zavolanlab/polyAsite_Atlas_3/"""+github_branch+"""/"""+organism_github_subfolder+"""/total/PAS."""+str(stringency_group)+'.'+strand+'.total.bb'+"""
        
"""
        f.write(bigbed)
        i=i+1

multiwig = """track PAS_average_RPM
visibility full
shortLabel PAS mean RPM
longLabel 2. PAS, average RPM across tissues
container multiWig
aggregate transparentOverlay
showSubtrackColorOnUi on
type bigWig 0 """+max_average+"""
viewLimits 0:"""+max_average+"""
autoScale on
maxHeightPixels 90:60:8
html https://raw.githubusercontent.com/zavolanlab/polyAsite_Atlas_3/"""+github_branch+"""/"""+organism_github_subfolder+"""/track_descriptions/PAS_average_RPM.html

"""
f.write(multiwig)

for stringency_group in stringency_groups:
    for strand in strands:
        strand_long_label = strands[strand]
        multiwig_subtrack = 'track PAS_'+str(stringency_group)+'_'+strand_long_label+'_strand'+"""
shortLabel RPM, """+str(stringency_group)+'%, '+strand+"""
longLabel PAS average RPM, for PAS with """+str(stringency_group)+"""% motif presence, on a ("""+strand+""") strand
parent PAS_average_RPM"""+"""
color """+('4,177,216' if strand=='+' else '255,68,51')+"""
type bigWig 0 """+max_average+"""
bigDataUrl https://raw.githubusercontent.com/zavolanlab/polyAsite_Atlas_3/"""+github_branch+"""/"""+organism_github_subfolder+"""/total/PAS."""+str(stringency_group)+'.'+strand_long_label+""".bw
html https://raw.githubusercontent.com/zavolanlab/polyAsite_Atlas_3/"""+github_branch+"""/"""+organism_github_subfolder+"""/track_descriptions/PAS_average_RPM.html

"""
        f.write(multiwig_subtrack)

i=3
for strand in strands:
    strand_long_label = strands[strand]
    multiwig = """track PAS_tissue_RPM_"""+strand_long_label+"""
visibility dense
shortLabel PAS tissue RPM ("""+strand+""") strand
longLabel """+str(i)+""". PAS tissue RPM ("""+strand+""") strand
container multiWig
aggregate none
showSubtrackColorOnUi on
type bigWig 0 """+max_tissue_level+"""
viewLimits 0:"""+max_tissue_level+"""
autoScale on
maxHeighPixels 120:60:8
html https://raw.githubusercontent.com/zavolanlab/polyAsite_Atlas_3/"""+github_branch+"""/"""+organism_github_subfolder+"""/track_descriptions/PAS_tissue_RPM_"""+strand_long_label+""".html

"""
    f.write(multiwig)
    i=i+1
    for tissue in tissues:
        subtrack = """track PAS_"""+tissue.replace(' ','_')+"""_RPM_"""+strand_long_label+"""_strand
shortLabel """+tissue+"""
longLabel PAS """+tissue+""" RPM ("""+strand+""") strand
parent PAS_tissue_RPM_"""+strand_long_label+"""
color """+('4,177,216' if strand=='+' else '255,68,51')+"""
type bigWig 0 """+max_tissue_level+"""
bigDataUrl https://raw.githubusercontent.com/zavolanlab/polyAsite_Atlas_3/"""+github_branch+"""/"""+organism_github_subfolder+"""/over_tissues/"""+'PAS.'+tissue+'.'+strand_long_label+""".bw
html https://raw.githubusercontent.com/zavolanlab/polyAsite_Atlas_3/"""+github_branch+"""/"""+organism_github_subfolder+"""/track_descriptions/PAS_tissue_RPM_"""+strand_long_label+""".html

"""
        f.write(subtrack)
f.close()